In [10]:
from pathlib import Path

# CHANGE THIS
ALIFIB_REPO = Path("/home/scanbot/alifib").resolve()

ALIFIB_FILE = ALIFIB_REPO / "examples" / "Category.ali"

print("repo:", ALIFIB_REPO)
print("file:", ALIFIB_FILE)
print("repo exists:", ALIFIB_REPO.exists())
print("file exists:", ALIFIB_FILE.exists())

repo: /home/scanbot/alifib
file: /home/scanbot/alifib/examples/Category.ali
repo exists: True
file exists: True


In [11]:
import subprocess

binary = ALIFIB_REPO / "target" / "release" / "alifib"

if not binary.exists():
    print("Building alifib...")
    subprocess.run(
        ["cargo", "build", "--release"],
        cwd=ALIFIB_REPO,
        check=True,
    )

print("Binary:", binary)
print("Exists:", binary.exists())

Binary: /home/scanbot/alifib/target/release/alifib
Exists: True


In [12]:
import json
import subprocess
from pathlib import Path


class AlifibClient:
    def __init__(self, repo_path):
        self.repo = Path(repo_path).resolve()
        self.binary = self.repo / "target" / "release" / "alifib"
        self.proc = None

        if not (self.repo / "Cargo.toml").exists():
            raise RuntimeError(f"Not an alifib repo: {self.repo}")

        if not self.binary.exists():
            subprocess.run(
                ["cargo", "build", "--release"],
                cwd=self.repo,
                check=True,
            )

        self.proc = subprocess.Popen(
            [str(self.binary), "serve"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )

    def _send(self, payload):
        if self.proc is None:
            raise RuntimeError("Process not started")
        if self.proc.stdin is None or self.proc.stdout is None:
            raise RuntimeError("Pipes not available")

        self.proc.stdin.write(json.dumps(payload) + "\n")
        self.proc.stdin.flush()

        line = self.proc.stdout.readline()
        if not line:
            err = ""
            if self.proc.stderr is not None:
                err = self.proc.stderr.read()
            raise RuntimeError(f"No response from alifib. stderr: {err}")

        resp = json.loads(line)
        if resp.get("status") != "ok":
            raise RuntimeError(resp.get("message", "Unknown alifib error"))

        return resp.get("data")

    def init(self, ali_file, type_name, source, target=None):
        payload = {
            "command": "init",
            "source_file": str(Path(ali_file).resolve()),
            "type_name": type_name,
            "source_diagram": source,
        }
        if target is not None:
            payload["target_diagram"] = target
        return self._send(payload)

    def step(self, choice):
        return self._send({
            "command": "step",
            "choice": choice,
        })

    def undo(self):
        return self._send({
            "command": "undo",
        })

    def show(self):
        return self._send({
            "command": "show",
        })

    def types(self):
        return self._send({
            "command": "types",
        })

    def type_info(self, name):
        return self._send({
            "command": "type",
            "type_name": name,
        })

    def shutdown(self):
        try:
            if self.proc is not None and self.proc.poll() is None:
                try:
                    self._send({"command": "shutdown"})
                except Exception:
                    pass
        finally:
            if self.proc is not None and self.proc.poll() is None:
                self.proc.terminate()
                self.proc.wait(timeout=2)